# TWSE PPO + SMC 訓練與驗證流程

這份 notebook 專注在目前正式版 pipeline。

流程目標：

1. 確認環境與 GPU。
2. 下載訓練資料。
3. 建立 SMC 數值化特徵。
4. 建立 Gymnasium 虛擬市場。
5. 訓練 PPO。
6. 下載驗證資料。
7. 建立驗證用 SMC 特徵。
8. 輸出驗證報表與圖表。

目前模型：

- Model 1：`0050.TW vs 2330.TW`
- Model 2：`0050.TW vs 2330.TW vs 2412.TW`


## 1. 環境確定

Kernel crash 通常是底層套件匯入失敗造成，例如 PyTorch、CUDA DLL、NVIDIA driver 或 Stable-Baselines3 相依套件。

因此這裡拆成幾個小步驟：

- 1.1 先確認 Jupyter kernel 本身可執行。
- 1.2 再確認 PyTorch 與 CUDA。
- 1.3 再匯入一般資料與 RL 套件。
- 1.4 最後匯入本專案模組。

如果 kernel 在某一格 crash，就代表問題集中在那一格的套件。


### 1.1 Kernel 基本檢查

這一格不匯入 GPU 或大型套件，只確認目前 Jupyter kernel 可以正常跑 Python。


In [ ]:
import sys
from pathlib import Path

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Working directory: {Path.cwd()}")


### 1.2 PyTorch 與裝置檢查

這一格會檢查目前是否可以使用 GPU。

- 如果 CUDA 可用，使用 `cuda:0`。
- 如果 CUDA 不可用，自動改用 `cpu`。
- 最後會顯示目前實際使用的裝置。

注意：使用 CPU 可以讓 notebook 繼續跑，但 PPO 訓練速度會明顯變慢。


In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda:0"
    device_name = torch.cuda.get_device_name(0)
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = "cpu"
    device_name = "CPU"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Using device: {DEVICE}")
print(f"Device name: {device_name}")


### 1.3 套件匯入檢查

這一格確認資料處理、繪圖與 Stable-Baselines3 可以正常匯入。


In [ ]:
import gymnasium as gym
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

print(f"Gymnasium: {gym.__version__}")
print(f"Pandas: {pd.__version__}")
print("Stable-Baselines3 PPO import: OK")


### 1.4 專案模組匯入檢查

這一格確認 notebook 可以讀到本專案的 pipeline 共用模組。


In [ ]:
from twse_pipeline_common import (
    BASKET_CONFIG,
    PAIR_CONFIG,
    TWTradingEnv,
    build_feature_frame,
    calculate_metrics,
    download_ohlcv,
    ensure_model_dir,
)

print("Project imports: OK")
print(f"Model 1: {PAIR_CONFIG.name} -> {PAIR_CONFIG.tickers}")
print(f"Model 2: {BASKET_CONFIG.name} -> {BASKET_CONFIG.tickers}")


## 2. 資料下載

這一步只下載原始 OHLCV 資料。

先不做特徵，也不建立環境，讓資料來源與時間區間保持清楚。


## 2.1 訓練資料的時間區間選擇

設定 in-sample 訓練區間與訓練參數。


In [ ]:
TRAIN_START = "2018-01-01"
TRAIN_END = "2024-12-31"
TOTAL_TIMESTEPS = 500_000
INITIAL_BALANCE = 1_000_000.0

MODEL_CONFIGS = [PAIR_CONFIG, BASKET_CONFIG]

print(f"Training period: {TRAIN_START} to {TRAIN_END}")
for config in MODEL_CONFIGS:
    print(f"{config.name}: {', '.join(config.tickers)} -> {config.model_path}")


In [ ]:
train_raw_frames = {}

for config in MODEL_CONFIGS:
    print(f"Downloading training data for {config.name}: {config.tickers}")
    train_raw_frames[config.name] = download_ohlcv(config.tickers, TRAIN_START, TRAIN_END)

for model_name, frames in train_raw_frames.items():
    print(f"\n{model_name}")
    for ticker, df in frames.items():
        print(f"  {ticker}: {df['date'].min()} -> {df['date'].max()}, rows={len(df)}")
        display(df.head())


## 3. SMC 資料正規化

這裡把 OHLCV 轉成 PPO 可讀的數值特徵。

目前的 SMC 特徵已經是模型可直接使用的尺度：

- `PD_Pos`：20 日 dealing range 位置，約 `0` 到 `1`。
- `OB_Dist`：距離 order block 的百分比距離。
- `FVG_Signal`：是否存在尚未回補 FVG，值為 `0` 或 `1`。
- `Spread_ZScore`：pair / basket 的 rolling z-score。

注意：這裡不額外做 mean/std scaling，避免 notebook 訓練與 `app.py`、`predict_pipeline.py` 推論輸入尺度不一致。


In [ ]:
def smc_feature_columns(df):
    tokens = ["PD_Pos", "OB_Dist", "FVG_Signal", "Spread_ZScore"]
    return [col for col in df.columns if any(token in col for token in tokens)]


train_feature_frames = {}

for config in MODEL_CONFIGS:
    feature_df = build_feature_frame(train_raw_frames[config.name], config)
    train_feature_frames[config.name] = feature_df

    feature_cols = smc_feature_columns(feature_df)
    print(f"\n{config.name}")
    print(f"rows={len(feature_df)}, smc_feature_count={len(feature_cols)}")
    display(feature_df[["date"] + feature_cols].head())


## 4. `gymnasium` 虛擬市場環境建立

把 SMC 特徵放進 `TWTradingEnv`。

環境負責：

- 現金與持倉。
- 買賣手續費。
- 交易稅。
- 10% 漲停限制。
- reward、drawdown 與交易紀錄。


In [ ]:
def make_vec_env(config, feature_df):
    return DummyVecEnv([
        lambda: TWTradingEnv(
            feature_df=feature_df,
            config=config,
            initial_balance=INITIAL_BALANCE,
        )
    ])


train_envs = {}

for config in MODEL_CONFIGS:
    env = make_vec_env(config, train_feature_frames[config.name])
    train_envs[config.name] = env
    raw_env = env.envs[0]
    print(
        f"{config.name}: obs_shape={raw_env.observation_space.shape}, "
        f"action_shape={raw_env.action_space.shape}"
    )


## 5. PPO 訓練

開始訓練 PPO agent。

訓練完成後，模型會儲存在 `model/saved/`。


In [ ]:
def train_ppo_model(config, env):
    policy_kwargs = dict(net_arch=dict(pi=[128, 128], vf=[128, 128]))
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=2e-4,
        n_steps=2048,
        batch_size=128,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.005,
        target_kl=0.03,
        policy_kwargs=policy_kwargs,
        verbose=1,
        device=DEVICE,
    )
    model.learn(total_timesteps=TOTAL_TIMESTEPS)
    ensure_model_dir(config.model_path)
    model.save(config.model_path)
    print(f"Saved model: {config.model_path}")
    return model


trained_models = {}

for config in MODEL_CONFIGS:
    print(f"\n=== Training {config.name} ===")
    trained_models[config.name] = train_ppo_model(config, train_envs[config.name])


## 6. 驗證資料的時間區間選擇

設定 out-of-sample 驗證區間。

這段資料只用來驗證，不會更新 PPO 權重。


In [ ]:
VAL_START = "2025-01-01"
VAL_END = "2026-05-01"

print(f"Validation period: {VAL_START} to {VAL_END}")


## 7. SMC 正規化

驗證資料使用與訓練資料相同的特徵生成邏輯。

這一步只建立 out-of-sample SMC 特徵，不會重新訓練，也不會修改模型權重。


In [ ]:
val_raw_frames = {}
val_feature_frames = {}

for config in MODEL_CONFIGS:
    print(f"Downloading validation data for {config.name}: {config.tickers}")
    raw_frames = download_ohlcv(config.tickers, VAL_START, VAL_END)
    feature_df = build_feature_frame(raw_frames, config)

    val_raw_frames[config.name] = raw_frames
    val_feature_frames[config.name] = feature_df

    feature_cols = smc_feature_columns(feature_df)
    print(f"\n{config.name}: rows={len(feature_df)}, smc_feature_count={len(feature_cols)}")
    display(feature_df[["date"] + feature_cols].head())


## 8. 驗證資料報表輸出

載入訓練完成的模型，逐日執行：

```python
model.predict(obs, deterministic=True)
```

驗證階段不會呼叫 `model.learn()`，因此不會更新 policy weights。


In [ ]:
def run_validation(config, feature_df):
    model = PPO.load(config.model_path, device=DEVICE)
    env = TWTradingEnv(
        feature_df=feature_df,
        config=config,
        initial_balance=INITIAL_BALANCE,
    )

    obs, _ = env.reset()
    done = False
    action_logs = []

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        action_values = action.reshape(-1).astype(float)
        row = {"date": env.df.loc[env.current_step, "date"], "action": float(action_values[0])}

        for idx, ticker in enumerate(config.tickers):
            if idx < len(action_values):
                row[f"action_{ticker}"] = float(action_values[idx])
        action_logs.append(row)

        obs, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    history = pd.DataFrame(env.history)
    actions = pd.DataFrame(action_logs)
    metrics = calculate_metrics(history, initial_balance=INITIAL_BALANCE)
    return history, actions, metrics


validation_results = []

for config in MODEL_CONFIGS:
    print(f"\n=== Validating {config.name} ===")
    history, actions, metrics = run_validation(config, val_feature_frames[config.name])
    validation_results.append({
        "config": config,
        "history": history,
        "actions": actions,
        "metrics": metrics,
    })

summary = pd.DataFrame([
    {
        "model": result["config"].name,
        "tickers": ", ".join(result["config"].tickers),
        "cumulative_return": result["metrics"]["cumulative_return"],
        "sharpe_ratio": result["metrics"]["sharpe_ratio"],
        "max_drawdown": result["metrics"]["max_drawdown"],
    }
    for result in validation_results
])

display(summary.style.format({
    "cumulative_return": "{:.2%}",
    "sharpe_ratio": "{:.2f}",
    "max_drawdown": "{:.2%}",
}))


In [ ]:
for result in validation_results:
    config = result["config"]
    history = result["history"]
    actions = result["actions"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=history["date"],
        y=history["net_worth"],
        mode="lines",
        name="Net Worth",
    ))
    fig.update_layout(
        title=f"{config.name} Validation Net Worth",
        yaxis_title="NTD",
        template="plotly_dark",
    )
    fig.show()

    action_cols = [col for col in actions.columns if col.startswith("action_")]
    fig_action = go.Figure()

    if action_cols:
        for col in action_cols:
            fig_action.add_trace(go.Scatter(
                x=actions["date"],
                y=actions[col],
                mode="lines",
                name=col.replace("action_", ""),
            ))
    else:
        fig_action.add_trace(go.Scatter(
            x=actions["date"],
            y=actions["action"],
            mode="lines",
            name="action",
        ))

    fig_action.update_layout(
        title=f"{config.name} Deterministic Actions",
        yaxis_title="Action [-1, 1]",
        template="plotly_dark",
    )
    fig_action.show()
